# 알약 탐지·분류 통합 추론

이 노트북은 학습된 탐지기와 분류기 `best.pt`를 불러와 원본 이미지에서 알약 위치와 세부 품목을 순서대로 예측하고 결과를 저장한다. 두 모델의 학습은 개별 학습 노트북에서 수행한다.

## 실행 환경

- **Colab:** 저장소, 두 checkpoint와 추론 이미지 경로를 준비한 뒤 실행한다.
- **로컬:** 저장소 루트에서 `uv sync --locked`를 실행하고 설정 셀에 추론 이미지 폴더를 입력한다. CUDA, MPS 또는 CPU 중 사용 가능한 장치를 자동으로 선택한다.

## 추론 실행 설정

설정 셀에서 탐지기·분류기 실행 이름, 입력 경로, batch 크기와 탐지 점수 임곗값을 지정한다. 결과는 `outputs/inference/<run_name>/`에 저장된다.

주요 결과물:

- `pill_predictions.json`: bbox, 품목 예측과 결합 점수
- `pill_recognition_errors.json`: 탐지 또는 분류 단계 오류
- `submission.csv`: Kaggle 제출 형식의 최종 예측
- `visualizations/`: 이미지별 최종 예측 시각화

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from configs.pill_recognition import PillRecognitionConfig
from pipeline import (
    plot_pill_predictions,
    prepare_pill_recognition_data,
    recognize_pills_from_config,
)

In [ ]:
run_name = "20261103-ssdlite320-resnet50-infer-test-full-01"
device = "auto"
show_progress = True

# tags = [
#     "task:inference",
#     "stage:full",
#     "detector:ssdlite320",
#     "classifier:resnet50",
#     "data:test-full",
#     "device:auto",
# ]

output_root = project_root / "outputs"
output_dir = output_root / "inference"
detector_run_name = "20261103-ssdlite320-det-mini-smoke-01"
classifier_run_name = "20261103-resnet50-cls-v1-baseline-01"
inference_factory_path = "data_loading.recognition:build_pill_recognition_inputs"
input_dir_value = "data/sprint_ai_project1_data/test_images"
input_dir = Path(input_dir_value).expanduser()
if not input_dir.is_absolute():
    input_dir = project_root / input_dir
batch_size = 1
score_threshold = 0.05

detector_run_dir = output_root / "detector" / detector_run_name
classifier_run_dir = output_root / "classifier" / classifier_run_name

detector_checkpoint_path = detector_run_dir / "best.pt"
classifier_checkpoint_path = classifier_run_dir / "best.pt"
detector_label_map_path = detector_run_dir / "label_map.json"
classifier_class_map_path = classifier_run_dir / "class_map.json"

config = PillRecognitionConfig(
    run_name=run_name,
    inference_factory_path=inference_factory_path,
    input_dir=input_dir,
    detector_checkpoint_path=detector_checkpoint_path,
    classifier_checkpoint_path=classifier_checkpoint_path,
    detector_label_map_path=detector_label_map_path,
    classifier_class_map_path=classifier_class_map_path,
    output_dir=output_dir,
    detector_name=detector_run_name,
    batch_size=batch_size,
    score_threshold=score_threshold,
    show_progress=show_progress,
    device=device,
)
run_dir = config.run_dir
print(f"====================\nrun_name: {run_name}\n====================\n")
print(
    f"device: {config.device}\n"
    f"input_dir: {config.input_dir}\n"
    f"run_dir: {config.run_dir}\n"
    f"detector_run: {detector_run_name}\n"
    f"classifier_run: {classifier_run_name}"
)
print(f"batch_size: {config.batch_size}")
print(f"score_threshold: {config.score_threshold}")

## 입력 준비 및 검증

`input_dir`에 추론할 이미지를 넣으면 기본 factory가 재귀적으로 읽는다. 외부 factory를 사용할 경우 `factory(config=config, batch_size=batch_size)` 호출을 지원하고 `images: list[Tensor]`와 `meta: list[dict]`를 반환해야 한다.

In [ ]:
inference_data = prepare_pill_recognition_data(config)
preview_batch = inference_data.preview_batch
device = inference_data.device
print(f"입력 검증 완료: device={device}, preview_batch={len(preview_batch['images'])}")

## 통합 추론 실행

두 checkpoint는 현재 label map 및 class map과 일치해야 한다. 모델은 같은 CUDA, MPS 또는 CPU 장치에서 순서대로 실행된다. 탐지 bbox를 이미지 경계로 보정해 crop하고 학습과 같은 padding·정규화를 적용한다. 최종 점수는 `detector_score × classifier_score`로 계산한다.

In [ ]:
recognition_result = recognize_pills_from_config(
    config,
    data=inference_data,
)
print("통합 추론과 artifact 저장 완료")

## 결과 artifact 및 제출 파일 확인

저장된 예측과 단계별 오류 수를 확인하고 Kaggle 제출 CSV의 컬럼, 행 수와 고유 ID를 검증한다.

In [ ]:
predictions_path = run_dir / "pill_predictions.json"
errors_path = run_dir / "pill_recognition_errors.json"
submission_path = run_dir / "submission.csv"
print(
    f"predictions={len(recognition_result.predictions)}, "
    f"detector_errors={len(recognition_result.detector_errors)}, "
    f"classifier_errors={len(recognition_result.classifier_errors)}"
)
print(f"saved: {predictions_path}")
print(f"saved: {errors_path}")
print(f"saved: {submission_path}")

In [ ]:
prediction_table = pd.DataFrame(
    [prediction.to_dict() for prediction in recognition_result.predictions]
)
display(prediction_table.head(20))

submission_columns = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]
submission_table = pd.read_csv(submission_path)
if list(submission_table.columns) != submission_columns:
    raise ValueError("submission.csv 컬럼이 Kaggle 제출 형식과 다릅니다.")
if len(submission_table) != len(recognition_result.predictions):
    raise ValueError("제출 행 수와 최종 예측 수가 다릅니다.")
if not submission_table["annotation_id"].is_unique:
    raise ValueError("annotation_id가 중복되었습니다.")
display(submission_table.head(20))

error_table = pd.DataFrame(
    [
        {"stage": "detector", **error.to_dict()}
        for error in recognition_result.detector_errors
    ]
    + [
        {"stage": "classifier", **error.to_dict()}
        for error in recognition_result.classifier_errors
    ]
)
display(error_table.head(20))

## 예측 시각화

첫 번째 추론 이미지에 최종 품목명, bbox와 결합 점수를 표시하고 PNG로 저장한다.

In [ ]:
first_image = preview_batch["images"][0]
first_image_id = str(preview_batch["meta"][0]["image_id"])
first_predictions = [
    prediction
    for prediction in recognition_result.predictions
    if prediction.image_id == first_image_id
]
figure = plot_pill_predictions(
    first_image,
    first_predictions,
    title=f"Pill recognition image_id={first_image_id}",
)
visualization_dir = run_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
figure